# HAKE-MER — Eval supplements (cardinality + lexicon ablation)

**No new hyperparameter search.** This notebook only:

1. Ensures DistilBERT checkpoints exist under `runs/` for Step~0, +M1, +M1+M2, M3 NRC, M3 SenticNet (trains missing stacks with the usual 4-epoch protocol).
2. Runs `./run_eval_supplements.sh` (test-only forward passes: `by_gold_cardinality`, M3 lexique zero/shuffle).
3. Downloads a zip of updated `metrics.json` + patched campaign JSON + LaTeX snippets.

Compare cardinality table to **+M1+M2: 0.505 ± 0.004** macro overall.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
a81f176


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.4 MB/s eta 0:00:00


In [4]:
from pathlib import Path

def need_train(glob_pat: str, n: int = 3) -> bool:
    ck = sorted(Path("runs").glob(glob_pat))
    print(glob_pat, len(ck), "checkpoints")
    return len(ck) < n

TRAIN_PLAN = [
    ("distilbert_base_uncased_seed*_baseline_plm/best_model.pt", "./run_baseline_plm_campaign.sh --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1/best_model.pt", "./run_m1_campaign.sh --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1_m2/best_model.pt", "./run_m1_m2_campaign.sh --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1_m2_m3_nrc/best_model.pt", "./run_m1_m2_m3_campaign.sh --lexicon nrc --backbone distilbert-base-uncased"),
    ("distilbert_base_uncased_seed*_m1_m2_m3_senticnet/best_model.pt", "./run_m1_m2_m3_campaign.sh --lexicon senticnet --backbone distilbert-base-uncased"),
]
for glob_pat, cmd in TRAIN_PLAN:
    if need_train(glob_pat):
        print("Training:", cmd)
        get_ipython().system(f"{cmd} {TRAIN_FLAGS}")

distilbert_base_uncased_seed*_baseline_plm/best_model.pt 0 checkpoints
Training: ./run_baseline_plm_campaign.sh --backbone distilbert-base-uncased
/bin/bash: line 1: ./run_baseline_plm_campaign.sh: No such file or directory
distilbert_base_uncased_seed*_m1/best_model.pt 0 checkpoints
Training: ./run_m1_campaign.sh --backbone distilbert-base-uncased
config.json: 100% 483/483 [00:00<00:00, 5.40MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 370kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 6.82MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 24.7MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 53.6MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:  82% 2.27M/2.77M [00:00<00:00, 6.02MB/s, 68.1kB/s  ]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 4.40MB/s,  268kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 4.46MB/s,  273kB/s  ]

simplified/validation-00000-of-00001.p

In [5]:
!ALLOW_EVAL=1 DEVICE=cuda ./run_eval_supplements.sh

Loading weights: 100% 100/100 [00:00<00:00, 16079.37it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 100/100 [00:00<00:00, 13326.25it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXP

In [6]:
from pathlib import Path
for snippet in ["cardinality_snippet.tex", "lexicon_ablation_snippet.tex"]:
    p = Path("reference/artifacts") / snippet
    print("---", snippet, "---")
    print(p.read_text(encoding="utf-8") if p.is_file() else "(missing)")

--- cardinality_snippet.tex ---
1 & {---} & {---} & {---} \\
\hline
2 & {---} & {---} & {---} \\
\hline
3 & {---} & {---} & {---} \\
\hline
4+ & {---} & {---} & {---} \\
\hline

--- lexicon_ablation_snippet.tex ---
M3 SenticNet (Normal) & $0{,}504 \pm 0{,}003$ \\
\hline
M3 SenticNet (Lexique nul) & {---} \\
\hline
M3 SenticNet (Lexique permuté) & {---} \\
\hline
M3 NRC (Normal) & $0{,}504 \pm 0{,}008$ \\
\hline
M3 NRC (Lexique nul) & {---} \\
\hline
M3 NRC (Lexique permuté) & {---} \\
\hline
M3+M4 NRC (Normal) & $0{,}510 \pm 0{,}003$ \\
\hline
M3+M4 NRC (Lexique nul) & $0{,}510 \pm 0{,}003$ \\
\hline
M3+M4 NRC (Lexique permuté) & $0{,}510 \pm 0{,}003$ \\
\hline



In [7]:
import zipfile
from google.colab import files
from pathlib import Path

out_name = "eval_supplements_distilbert.zip"
zip_path = Path(f"/content/{out_name}")
campaigns = [
    "baseline_plm_distilbert_base_uncased_campaign.json",
    "m1_distilbert_base_uncased_campaign.json",
    "m1_m2_distilbert_base_uncased_campaign.json",
    "m1_m2_m3_nrc_distilbert_base_uncased_campaign.json",
    "m1_m2_m3_senticnet_distilbert_base_uncased_campaign.json",
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in campaigns:
        p = Path("reference/artifacts") / name
        if p.is_file():
            zf.write(p, f"campaigns/{name}")
    for snippet in ("cardinality_snippet.tex", "lexicon_ablation_snippet.tex"):
        p = Path("reference/artifacts") / snippet
        if p.is_file():
            zf.write(p, snippet)
    for m in sorted(Path("runs").glob("distilbert_base_uncased_seed*/metrics.json")):
        zf.write(m, f"metrics/{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download eval_supplements_distilbert.zip (33.1 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Integrity

1. **File → Download → .ipynb** → `reference/training_records/step_eval_supplements/colab/`
2. Send **zip** + executed notebook for repo integration (`merge_archived_metrics_into_campaigns.py` + commit snippets).